::: {.sb-lesson-links}
[Exercises →](../exercises/obi-wan-kenobi.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/04-nlp.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-nlp-basics.ipynb){download="lesson-nlp-basics.ipynb"}
:::

Text is everywhere: reviews, emails, articles, messages. But a model only understands numbers. This lesson shows how to clean a text, cut it into words, and turn it into a table of numbers (bag of words and TF-IDF), then how to measure the similarity between two texts.

In [ ]:
import re

import numpy as np
import pandas as pd
import plotly.express as px
import spacy
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from spacy.lang.en.stop_words import STOP_WORDS

import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"

nlp = spacy.load("en_core_web_sm")   # the small English model of spaCy

## What is NLP? 🦜

**Natural language processing** (NLP) studies how computers can work with human language ("natural" language, as opposed to programming languages). Some common applications:

* spell checking, keyword and synonym search;
* extracting information from texts (prices, dates, names of people or companies…);
* classification: sentiment analysis (is this review positive?), spam detection;
* **topic modeling**: finding the themes of a collection of texts, without labels (an unsupervised task, and the subject of the next section);
* translation, speech assistants, chatbots.

### Why is text a special kind of data?

Most data that data scientists handle is quantitative (a price, an age) or categorical (a colour, a country). Text is neither, and it brings its own problems:

* **Text is not numerical.** We must invent a way to turn words into numbers.
* **Language is ambiguous.** "I saw her duck" can talk about an animal or a movement. The meaning depends on the context, on common sense, on culture.
* **Written and spoken language differ.** Abbreviations, typos, emojis, slang…
* **Order matters.** Words do not come in a random order: "the dog bites the man" and "the man bites the dog" use the same words but do not mean the same thing.



::: {.callout-tip title="Intuition"}
A model needs a table: one row per observation, one column per feature. For text, the observation is a **document** (a sentence, a review, a paragraph, a chapter: you choose), and the features will be the **words**. Everything in this lesson is about building that table, while losing as little meaning as possible.
:::



## Choosing the unit of analysis: n-grams 🏘️

Before counting anything, you must decide what a "unit" of language is. You can work at the level of the character, the word, the group of words, the sentence…

The most common choice is the **n-gram**: a group of $n$ consecutive words. For "the cat eats the fish":

* 1-grams (unigrams): "the", "cat", "eats", "the", "fish";
* 2-grams (bigrams): "the cat", "cat eats", "eats the", "the fish".

#### See it with data 🔬: unigrams lose the order, bigrams keep some of it

Scikit-learn's `CountVectorizer` cuts texts into n-grams and counts them. Let's compare the two sentences about the dog and the man.

In [ ]:
sentences = ["the dog bites the man", "the man bites the dog"]

for n in [1, 2]:
    vectorizer = CountVectorizer(ngram_range=(n, n))
    counts = vectorizer.fit_transform(sentences).toarray()
    print(f"{n}-grams:")
    print(pd.DataFrame(counts, columns=vectorizer.get_feature_names_out(), index=sentences), "\n")

1-grams:
                       bites  dog  man  the
the dog bites the man      1    1    1    2
the man bites the dog      1    1    1    2 

2-grams:
                       bites the  dog bites  man bites  the dog  the man
the dog bites the man          1          1          0        1        1
the man bites the dog          1          0          1        1        1 



**What this shows:** with unigrams, both sentences give exactly the same counts (`bites` 1, `dog` 1, `man` 1, `the` 2): the model cannot tell them apart. With bigrams, they differ: only the first one contains "dog bites", only the second one contains "man bites". Bigrams keep a bit of the word order, at the price of many more columns (bigrams are much more varied than single words).

## Parse trees 🌳

Another way to use the order of words is to analyse the **grammar** of the sentence. A sentence follows rules: for example, a sentence (S) can be a noun phrase (NP) followed by a verb phrase (VP). Applying these rules again and again cuts the sentence into a **parse tree**:

![Parse tree of the sentence 'The cat eats the fish': the sentence S splits into a noun phrase (The cat) and a verb phrase; the verb phrase splits into the verb (eats) and a second noun phrase (the fish)](../img/nlp_parse_tree.svg)

Parse trees work well on short, precise and factual sentences, like the instructions given to a voice assistant ("set an alarm at seven"). On long, complex sentences, they become hard to use. In the next lesson, spaCy will draw a similar structure for us (a dependency tree).

In the rest of this lesson, we take the simplest and most common road in text mining: **forget the grammar and count the words**.

## The preprocessing pipeline 🚧

Raw text is messy. Before counting words, we clean it in a few steps:

![The preprocessing pipeline: raw text, cleaning (lower case, no punctuation), tokenization, removal of stop words, lemmatization, and vectorization into numbers, shown on the sentence 'The cats are chasing mice!'](../img/nlp_pipeline.svg)

Not every project needs every step: you choose them according to your goal. Let's look at each one.

### Lower case and punctuation

For Python, `"Learn"` and `"learn"` are two different strings, although they are the same word for us. So we usually put the whole text in lower case. Punctuation is rarely useful for text mining, so we usually remove it too.

Both are done with **regular expressions** (the `re` module): a small language to describe patterns in a text. Below, `[^a-z0-9 ]` means "any character that is not a lower case letter, a digit or a space". The site [regexr.com](https://regexr.com/) is a nice playground to build and test regular expressions.

In [ ]:
text = "Hello World! Don't do this. I am Superman, wherever you are in the world..."

lower = text.lower()
clean = re.sub(r"[^a-z0-9 ]", " ", lower)    # replace every other character by a space
clean = re.sub(r" +", " ", clean).strip()     # collapse multiple spaces
print(lower)
print(clean)

hello world! don't do this. i am superman, wherever you are in the world...
hello world don t do this i am superman wherever you are in the world


**What this shows:** "World" and "world" are now the same word. But the rule is blind: "Don't" became "don t", two pieces that make no sense alone. This is why we use a real tokenizer for the next step.

### Tokenization

**Tokenizing** means cutting a text into units called **tokens** (usually words and punctuation marks). Splitting on spaces is a first idea, but it is not enough.

#### See it with data 🔬: `split()` vs. spaCy's tokenizer

In [ ]:
sentence = "Don't pay $9.99 for this, it's free in the U.S.A.!"
print("split():", sentence.split())
print("spaCy:  ", [token.text for token in nlp(sentence)])

split(): ["Don't", 'pay', '$9.99', 'for', 'this,', "it's", 'free', 'in', 'the', 'U.S.A.!']
spaCy:   ['Do', "n't", 'pay', '$', '9.99', 'for', 'this', ',', 'it', "'s", 'free', 'in', 'the', 'U.S.A.', '!']


**What this shows:** `split()` glues the punctuation to the words (`'this,'`, `'U.S.A.!'`) and keeps `"$9.99"` as one block. spaCy knows the rules of English: it separates "Do" and "n't", "it" and "'s", the `$` sign from the price `9.99`, and keeps "U.S.A." as one token. Each language has its own rules, so spaCy has one model per language.

### Stop words

**Stop words** are the very frequent words that carry little meaning on their own: articles, pronouns, linking words ("the", "a", "and", "of", "is"…). They are everywhere, so they hide the words that really characterize a text. We usually remove them.

#### See it with data 🔬: how much of a text is made of stop words?

spaCy has a list of English stop words. Let's measure their share in a short paragraph.

In [ ]:
paragraph = ("The weather was cold, so we stayed at home and we cooked a big soup with the vegetables "
             "that we had bought at the market. It was not the best soup of the year, but it was warm.")
words = [token.text.lower() for token in nlp(paragraph) if token.is_alpha]   # words only, no punctuation
stop = [w for w in words if w in STOP_WORDS]
print(f"spaCy knows {len(STOP_WORDS)} English stop words")
print(f"{len(stop)} of the {len(words)} words of the paragraph are stop words ({len(stop) / len(words):.0%})")
print("Kept:", [w for w in words if w not in STOP_WORDS])

spaCy knows 326 English stop words
24 of the 38 words of the paragraph are stop words (63%)
Kept: ['weather', 'cold', 'stayed', 'home', 'cooked', 'big', 'soup', 'vegetables', 'bought', 'market', 'best', 'soup', 'year', 'warm']


**What this shows:** 24 of the 38 words (63%) are stop words. The 14 words that remain (weather, cold, soup, market…) are enough to guess what the paragraph is about. But look closely: "not" was removed too, because it is in spaCy's list. "It was not the best soup" became "best soup", the opposite meaning! For sentiment analysis, keep the negations. A list of stop words is a choice, not a law.

Two other kinds of words are often removed, depending on the project:

* **Words common to the whole corpus.** If all your texts are about travelling in Italy, the words "Italy" and "travel" are everywhere and do not help to tell the texts apart. TF-IDF (below) lowers their weight automatically, and `max_df` (next lesson) can remove them.
* **Rare words.** A word that appears in one text only (often a typo) mostly adds noise. `min_df` (next lesson) removes them. Some projects also correct typos with a spell checker.

### Stemming vs. lemmatization

Many words are variations of a common root: "cat" and "cats", "study", "studies" and "studied". These variations are called **inflections**. If we count them as different words, we split the information. Two techniques group them:

* **Stemming** chops the end of the words with simple rules. It is fast but crude: the result is not always a real word ("ponies" → "poni").
* **Lemmatization** replaces each word by its **lemma**, the form you would find in a dictionary. It uses a vocabulary and the role of the word in the sentence: "mice" → "mouse", "was" → "be". It is slower, but much cleaner.

#### See it with data 🔬: a toy stemmer vs. spaCy's lemmas

spaCy does not do stemming. Below, a toy stemmer with a few rules of the classic Porter stemmer (the real one, available in the NLTK library, has many more rules).

In [ ]:
def toy_stem(word):
    """Chop common English endings (a tiny part of the Porter stemmer)."""
    for ending, replacement in [("sses", "ss"), ("ies", "i"), ("ss", "ss"), ("s", ""), ("ing", ""), ("ed", "")]:
        if word.endswith(ending):
            return word[: len(word) - len(ending)] + replacement
    return word

words = ["cats", "caresses", "ponies", "studies", "studying", "studied", "running", "mice", "was", "better"]
pd.DataFrame({
    "word": words,
    "stem (toy stemmer)": [toy_stem(w) for w in words],
    "lemma (spaCy)": [nlp(w)[0].lemma_ for w in words],
})

,word,stem (toy stemmer),lemma (spaCy)
0,cats,cat,cat
1,caresses,caress,caress
2,ponies,poni,pony
3,studies,studi,study
4,studying,study,study
5,studied,studi,study
6,running,runn,run
7,mice,mice,mouse
8,was,wa,be
9,better,better,well


**What this shows:** both methods agree on easy words ("cats" → "cat", "caresses" → "caress"). Elsewhere, the toy stemmer produces non-words ("poni", "studi", "runn", "wa"), and it even splits one word into two columns: "studying" → "study", but "studies" and "studied" → "studi". It cannot handle irregular forms: "mice" and "was" do not change. spaCy gives a dictionary form each time: "pony", "mouse", "be", and "study" for the three forms of *study*. Lemmatizers are not perfect either: for "better", spaCy's small model gives "well".



::: {.callout-tip title="Intuition"}
Stemming is like cutting a word with scissors at a fixed place; lemmatization is like looking it up in a dictionary. For counting words, the goal is the same: all the forms of one word should fall into **one** column of our table.
:::



## From text to numbers 🔢

Our texts are now clean lists of words. The last step of the pipeline turns them into numbers.

### Bag of words

The **bag of words** forgets the order of the words and only keeps how many times each word appears. All the words of the corpus form the **vocabulary**; each word becomes a column, each document a row. The result is called the **document-term matrix**.

We'll use a tiny corpus of three documents for the rest of the lesson.

In [ ]:
corpus = [
    "the cat eats fish",
    "the cat chases fish",
    "the dog eats meat",
]
docs = ["d1", "d2", "d3"]

count_vectorizer = CountVectorizer()
counts = count_vectorizer.fit_transform(corpus)
tf = pd.DataFrame(counts.toarray(), columns=count_vectorizer.get_feature_names_out(), index=docs)
tf

,cat,chases,dog,eats,fish,meat,the
d1,1,0,0,1,1,0,1
d2,1,1,0,0,1,0,1
d3,0,0,1,1,0,1,1


**What this shows:** the vocabulary has 7 words (in alphabetical order), so each document becomes a vector of 7 numbers. "the" appears in every document, "fish" in d1 and d2, "meat" only in d3. The counts already say that d1 and d2 talk about the same things (a cat, fish). We keep "the" on purpose, to see what TF-IDF does with a word that is everywhere.

### Term frequency (TF)

The **term frequency** of a word in a document measures how often it appears:

$$
\text{tf}(t, d) = \frac{\text{number of times the term } t \text{ appears in } d}{\text{number of words in } d}
$$

Dividing by the length lets us compare a short text with a long one. For example, for the sentence *"The black cat eats fish but does not eat other black cats."*, after lemmatization and removal of the stop words:

In [ ]:
sentence = "The black cat eats fish but does not eat other black cats."
lemmas = [token.lemma_.lower() for token in nlp(sentence) if token.is_alpha and not token.is_stop]
occurrences = pd.Series(lemmas).value_counts()
pd.DataFrame({"occurrences": occurrences, "term frequency": occurrences / occurrences.sum()})

,occurrences,term frequency
black,2,0.285714
cat,2,0.285714
eat,2,0.285714
fish,1,0.142857


**What this shows:** the stop words "the", "but", "does", "not" and "other" are gone, and "cats" and "eats" became "cat" and "eat", so these words are now counted twice. 7 words remain: "black", "cat" and "eat" appear twice (tf = 2/7 ≈ 0.286) and "fish" once (1/7 ≈ 0.143). The term frequencies sum to 1.

### Inverse document frequency (IDF)

A word can be frequent in a document just because it is frequent everywhere ("the", or "Italy" in a corpus about Italy). Such a word does not help to tell the documents apart. The **inverse document frequency** measures how rare a word is in the corpus:

$$
\text{idf}(t) = \ln\left(\frac{n}{\text{df}(t)}\right)
$$

where $n$ is the number of documents and $\text{df}(t)$, the **document frequency**, is the number of documents that contain the term $t$.

* A word present in **every** document: $\text{df} = n$, so $\text{idf} = \ln(1) = 0$.
* A word present in **one** document only: $\text{idf} = \ln(n)$, the maximum.

### TF-IDF

**TF-IDF** multiplies the two:

$$
\text{tf-idf}(t, d) = \text{tf}(t, d) \times \text{idf}(t)
$$

A word gets a high weight in a document when it is **frequent in this document** and **rare in the others**: it is a word that characterizes the document.



::: {.callout-tip title="Intuition"}
Think of a detective reading three letters. The word "the" appears in all of them: it says nothing about who wrote which letter. The word "meat" appears in only one letter: that is a clue! TF-IDF gives each word a weight that measures how much of a clue it is for each document.
:::



#### See it with data 🔬: TF-IDF by hand

Let's compute every step on our three documents, then compare with scikit-learn. First the document frequency and the idf of each word.

In [ ]:
n = len(corpus)
df_t = (tf > 0).sum()                      # number of documents that contain each word
idf = np.log(n / df_t)
table = pd.DataFrame({"df": df_t, "idf = ln(n/df)": idf})
print(table.round(3), "\n")

tf_textbook = tf.div(tf.sum(axis=1), axis=0)   # count / number of words in the document
print("tf = count / length:")
print(tf_textbook.round(3), "\n")
print("tf-idf = tf x idf:")
print((tf_textbook * idf).round(3))

        df  idf = ln(n/df)
cat      2           0.405
chases   1           1.099
dog      1           1.099
eats     2           0.405
fish     2           0.405
meat     1           1.099
the      3           0.000 

tf = count / length:
     cat  chases   dog  eats  fish  meat   the
d1  0.25    0.00  0.00  0.25  0.25  0.00  0.25
d2  0.25    0.25  0.00  0.00  0.25  0.00  0.25
d3  0.00    0.00  0.25  0.25  0.00  0.25  0.25 

tf-idf = tf x idf:
      cat  chases    dog   eats   fish   meat  the
d1  0.101   0.000  0.000  0.101  0.101  0.000  0.0
d2  0.101   0.275  0.000  0.000  0.101  0.000  0.0
d3  0.000   0.000  0.275  0.101  0.000  0.275  0.0


**What this shows:** "the" is in the 3 documents, so its idf is $\ln(3/3) = 0$ and its tf-idf is 0 everywhere, although its tf (0.25) is as high as the tf of any other word. The words in two documents (cat, eats, fish) get $\ln(3/2) = 0.405$, and the words in one document only (chases, dog, meat) get $\ln(3) = 1.099$. So in d2, "chases" (0.275) weighs more than "cat" and "fish" (0.101): it is the word that makes d2 different from d1.

### What scikit-learn really computes

Scikit-learn's `TfidfVectorizer` follows the same idea with three small changes:

1. **tf** is the raw count (not divided by the length of the document).
2. **Smooth idf**: it computes
$$
\text{idf}(t) = \ln\left(\frac{1 + n}{1 + \text{df}(t)}\right) + 1
$$
The "+1" inside the fraction acts as if one extra document contained every word once, to avoid dividing by zero. The "+1" at the end keeps a small weight for the words present everywhere, instead of removing them completely.

3. **L2 normalization**: each row (each document) is divided by its length (its Euclidean norm), so that every document vector has a length of 1:
$$
\vec{v}_d \leftarrow \frac{\vec{v}_d}{\lVert \vec{v}_d \rVert} = \frac{\vec{v}_d}{\sqrt{\sum_t v_{d,t}^2}}
$$

Let's do these three steps by hand.

In [ ]:
idf_smooth = np.log((1 + n) / (1 + df_t)) + 1        # step 2
table["sklearn idf"] = idf_smooth
print(table.round(3), "\n")

raw = tf * idf_smooth                                   # steps 1 and 2: raw count x smooth idf
norms = np.sqrt((raw**2).sum(axis=1))                   # length of each row
by_hand = raw.div(norms, axis=0)                        # step 3: each row now has length 1
print("TF-IDF by hand:")
print(by_hand.round(3))

        df  idf = ln(n/df)  sklearn idf
cat      2           0.405        1.288
chases   1           1.099        1.693
dog      1           1.099        1.693
eats     2           0.405        1.288
fish     2           0.405        1.288
meat     1           1.099        1.693
the      3           0.000        1.000 

TF-IDF by hand:
      cat  chases    dog   eats   fish   meat    the
d1  0.527   0.000  0.000  0.527  0.527  0.000  0.409
d2  0.480   0.632  0.000  0.000  0.480  0.000  0.373
d3  0.000   0.000  0.584  0.445  0.000  0.584  0.345


In [ ]:
tfidf_vectorizer = TfidfVectorizer()        # default: smooth_idf=True, norm="l2"
tfidf = pd.DataFrame(tfidf_vectorizer.fit_transform(corpus).toarray(),
                     columns=tfidf_vectorizer.get_feature_names_out(), index=docs)
print("TF-IDF by scikit-learn:")
print(tfidf.round(3))
print("\nSame idf as by hand?   ", np.allclose(tfidf_vectorizer.idf_, idf_smooth))
print("Same matrix as by hand?", np.allclose(tfidf, by_hand))

# dividing the counts by the length of the document first changes nothing after the L2 normalization
textbook = tf_textbook * idf_smooth
print("Same with tf = count / length?", np.allclose(textbook.div(np.sqrt((textbook**2).sum(axis=1)), axis=0), tfidf))

TF-IDF by scikit-learn:
      cat  chases    dog   eats   fish   meat    the
d1  0.527   0.000  0.000  0.527  0.527  0.000  0.409
d2  0.480   0.632  0.000  0.000  0.480  0.000  0.373
d3  0.000   0.000  0.584  0.445  0.000  0.584  0.345

Same idf as by hand?    True
Same matrix as by hand? True
Same with tf = count / length? True


**What this shows:** the two matrices are identical, and so are the idf values. With the smooth idf, "the" is not removed: its idf is 1.000, the smallest value, against 1.288 for the words in two documents and 1.693 for the words in one document. So "the" gets the lowest weight of every document (0.409 in d1, against 0.527 for cat, eats and fish). The last line shows that dividing the counts by the length of the document (the textbook tf) changes nothing: the L2 normalization divides all the values of a row by the same number anyway.

#### See it with data 🔬: the idf curve

Let's draw the two idf formulas for a corpus of $n = 100$ documents, for every possible document frequency.

In [ ]:
n_docs = 100
df_values = np.arange(1, n_docs + 1)
curves = pd.DataFrame({
    "document frequency": df_values,
    "ln(n/df) (textbook)": np.log(n_docs / df_values),
    "ln((1+n)/(1+df)) + 1 (scikit-learn)": np.log((1 + n_docs) / (1 + df_values)) + 1,
})
fig = px.line(curves, x="document frequency", y=curves.columns[1:],
              title="A word present in many documents gets a small idf (n = 100 documents)")
fig.update_layout(yaxis_title="idf", legend_title="")
fig.show()
print(curves.iloc[[0, 9, 49, 99]].round(2).to_string(index=False))

 document frequency  ln(n/df) (textbook)  ln((1+n)/(1+df)) + 1 (scikit-learn)
                  1                 4.61                                 4.92
                 10                 2.30                                 3.22
                 50                 0.69                                 1.68
                100                 0.00                                 1.00


**What this shows:** both curves go down: the more documents contain a word, the smaller its idf. A word in 1 document out of 100 gets 4.61 (textbook) or 4.92 (scikit-learn); a word in half of the documents gets 0.69 or 1.68; a word in all 100 documents gets 0, or 1.00 with scikit-learn's smooth version. Because of the logarithm, the idf drops fast for the first documents (from 4.61 to 2.30 between 1 and 10 documents), then slowly.

## Cosine similarity: how close are two texts? 📐

Each document is now a vector. To compare two documents, we compare their vectors. For texts, the natural measure is the **cosine similarity**: the cosine of the angle between the two vectors.

$$
\cos(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{\lVert \vec{a} \rVert \, \lVert \vec{b} \rVert} = \frac{\sum_t a_t b_t}{\sqrt{\sum_t a_t^2} \, \sqrt{\sum_t b_t^2}}
$$

* $\cos = 1$: the vectors point in the same direction, the texts use the same words in the same proportions.
* $\cos = 0$: the vectors are perpendicular, the texts share no word.

TF-IDF values are never negative, so the cosine is always between 0 and 1. And since scikit-learn's TF-IDF rows already have a length of 1, the cosine is simply the dot product $\sum_t a_t b_t$.

#### See it with data 🔬: the cosine, by hand and with scikit-learn

In [ ]:
d1, d2 = tfidf.loc["d1"], tfidf.loc["d2"]
print(f"By hand, cos(d1, d2) = sum of the products = {(d1 * d2).sum():.3f}")

similarity = pd.DataFrame(cosine_similarity(tfidf), index=docs, columns=docs)
fig = px.imshow(similarity, text_auto=".2f", color_continuous_scale="Blues", zmin=0, zmax=1,
                title="Cosine similarity between the 3 documents (TF-IDF)")
fig.show()
print(similarity.round(3))

By hand, cos(d1, d2) = sum of the products = 0.659


       d1     d2     d3
d1  1.000  0.659  0.375
d2  0.659  1.000  0.129
d3  0.375  0.129  1.000


**What this shows:** the sum of the products of d1 and d2 gives 0.659, the same value as scikit-learn's matrix. d1 and d2 are the most similar pair (they share "the", "cat" and "fish"). d1 and d3 come next (0.375: they share "the" and "eats"). d2 and d3 are the least similar (0.129): they only share "the", the word with the smallest weight. The diagonal is 1: each document is identical to itself.

#### See it with data 🔬: why not the Euclidean distance?

Let's add a fourth document: d1 written three times. It says exactly the same thing as d1, only longer.

In [ ]:
corpus4 = corpus + ["the cat eats fish. the cat eats fish. the cat eats fish."]
docs4 = docs + ["d1 x 3"]
counts4 = CountVectorizer().fit_transform(corpus4)

print("Euclidean distance from d1 (on the counts):")
print(pd.Series(euclidean_distances(counts4[0], counts4).ravel(), index=docs4).round(2), "\n")
print("Cosine similarity with d1 (on the counts):")
print(pd.Series(cosine_similarity(counts4[0], counts4).ravel(), index=docs4).round(2))

Euclidean distance from d1 (on the counts):
d1        0.00
d2        1.41
d3        2.00
d1 x 3    4.00
dtype: float64 

Cosine similarity with d1 (on the counts):
d1        1.00
d2        0.75
d3        0.50
d1 x 3    1.00
dtype: float64


**What this shows:** for the Euclidean distance, "d1 x 3" is the farthest document from d1 (4.00), farther than d3 (2.00), which talks about a dog and meat! Its counts are three times bigger, so its vector is far away. Its cosine similarity with d1 is 1.00: same words, same proportions, same direction. The cosine also ranks the two other documents in a sensible order: d2 (0.75) before d3 (0.50).



::: {.callout-tip title="Intuition"}
The cosine looks at the **direction** of the vectors, not at their length. The direction says *which words* a text uses and in what proportions (its subject); the length mostly says *how long* the text is. Two articles on the same subject, one short and one long, point in the same direction.
:::



This is why the cosine similarity is the usual way to compare texts: search engines use it to rank documents for a query, and it is the basis of the clustering of texts. Note that on L2-normalized vectors (like scikit-learn's TF-IDF rows), the two measures are linked: $\lVert \vec{a} - \vec{b} \rVert^2 = 2 - 2\cos(\vec{a}, \vec{b})$. So a K-means on TF-IDF vectors groups texts by cosine similarity, as we'll use in the next section and in the project.

## Summary 📝

* A model needs numbers: NLP turns each **document** into a vector, with one column per word (or n-gram) of the vocabulary.
* The usual **pipeline**: lower case and remove punctuation, **tokenize**, remove **stop words**, **lemmatize** (or stem), then **vectorize**. Choose the steps for your goal: removing "not" can flip the meaning of a review.
* The **bag of words** counts the words and forgets their order; bigrams keep a bit of it.
* **TF-IDF** = term frequency × inverse document frequency: a word weighs a lot in a document when it is frequent in it and rare elsewhere. A word present everywhere gets the smallest idf.
* Scikit-learn uses raw counts, a **smooth idf** $\ln\frac{1+n}{1+\text{df}}+1$, and **L2-normalizes** each row.
* The **cosine similarity** compares the direction of two vectors, so it ignores the length of the texts: it is the natural similarity for texts.

## Resources 📚

* [Stemming and lemmatization](https://nlp.stanford.edu/IR-book/html/htmledition/stemming-and-lemmatization-1.html), in *Introduction to Information Retrieval* (Manning, Raghavan, Schütze)
* [TF-IDF](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) on Wikipedia
* [Text feature extraction](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction) in the scikit-learn user guide

::: {.sb-next}
**[Next: NLP with spaCy and scikit-learn →](../lessons/workflow.ipynb)**
:::